# Artificial Intelligence — Lab 11
## Introduction to Machine Learning: Training, Prediction, and Evaluation

**Course Learning Outcome — CLO6**  
Categorize and apply introductory concepts from advanced AI subdomains, including Machine Learning.

**Environment:** Python 3 / Jupyter Notebook  
**Libraries:** `numpy`, `matplotlib`, `scikit-learn`  
**Submission:** completed notebook containing predictions, code, experimental results, justifications, debugging answers, and reflection.

> **Assessment principle:** Working code is only part of the evidence. Most marks come from your ability to **explain the learning pipeline, justify train/test separation, interpret model errors, distinguish training from inference, and reason about generalization**.

## Lab at a Glance

| Stage | Suggested time | What you will do |
|---|---:|---|
| 1. ML problem formulation | 15 min | Identify features, targets, examples, and task type |
| 2. Explore the dataset | 15 min | Inspect class balance and feature distributions |
| 3. Train / validation / test split | 20 min | Build a valid evaluation pipeline |
| 4. Train an initial classifier | 20 min | Fit a decision tree and make validation predictions |
| 5. Model selection | 25 min | Compare tree depths using validation performance |
| 6. Final test evaluation | 15 min | Evaluate the selected model once on unseen test data |
| 7. Debugging, variation & reflection | 10 min | Diagnose leakage and defend conclusions |

> **Main idea:** Machine Learning learns a mapping from data. **Training data fits the model, validation data guides model selection, and test data is reserved for final evaluation.**

## Learning Objectives

By the end of this lab, you should be able to:

1. distinguish **features** from **targets/labels**;
2. identify a supervised classification problem;
3. explain the different roles of **training, validation, and test data**;
4. train a simple classifier using `scikit-learn`;
5. distinguish **training** from **inference/prediction**;
6. compute and interpret accuracy;
7. interpret a confusion matrix;
8. explain underfitting, overfitting, and generalization conceptually;
9. select a model using validation data and evaluate it once on test data;
10. diagnose common Machine Learning workflow errors such as leakage and test-set tuning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

print("Lab 11 environment ready.")

# Part I — Machine Learning as an AI Approach

In supervised learning, we are given examples

$$
(x_i,y_i)
$$

where:

- $x_i$ is an input represented by **features**;
- $y_i$ is the correct **target/label**.

A learning algorithm uses training examples to construct a model

$$
\hat{y}=f_\theta(x)
$$

that predicts a label for a new input.

In this lab, the task is **classification**.

## Task 1.1 — Identify the Learning Components

For a flower-classification system, suppose the input measurements are:

- sepal length;
- sepal width;
- petal length;
- petal width.

The target is the flower species.

Answer:

1. What are the **features**?
2. What is the **target**?
3. Is this supervised or unsupervised learning?
4. Is this classification or regression?
5. What would one training example contain?

**Your answers:**

## Task 1.2 — Training vs. Inference

Explain the difference between:

### Training

$$
\text{data} \rightarrow \text{learning algorithm} \rightarrow \text{model}
$$

and

### Inference

$$
\text{new input} + \text{trained model} \rightarrow \text{prediction}
$$

Then answer:

> Why should the model not be retrained every time we classify one new flower?

**Your answer:**

# Part II — Load and Inspect the Iris Dataset

The Iris dataset contains 150 flowers from three classes:

- `setosa`
- `versicolor`
- `virginica`

Each example has four numeric features.

In [ ]:
iris = load_iris()

X = iris.data
y = iris.target

feature_names = iris.feature_names
class_names = iris.target_names

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features:", feature_names)
print("Classes:", class_names)

## Task 2.1 — Interpret Dataset Dimensions

From the output:

1. How many examples are in the dataset?
2. How many features describe each example?
3. How many classes exist?
4. What does one row of `X` represent?
5. What does the corresponding value in `y` represent?

**Your answers:**

In [ ]:
print("First five feature rows:")
print(X[:5])

print("\nFirst five labels:")
print(y[:5])

print("\nCorresponding class names:")
print([class_names[label] for label in y[:5]])

## Task 2.2 — Predict Class Balance

Before running the next cell, predict:

- Are the three classes likely to be approximately balanced?
- If a classifier guessed only the most frequent class every time, what accuracy would you expect approximately?

**Your prediction:**

In [ ]:
unique, counts = np.unique(y, return_counts=True)

print("Class counts:")
for label, count_value in zip(unique, counts):
    print(class_names[label], "->", count_value)

majority_baseline = counts.max() / counts.sum()
print("\nMajority-class baseline accuracy:", majority_baseline)

## Task 2.3 — Why the Baseline Matters

Answer:

1. What is the majority-class baseline accuracy?
2. Why is a model accuracy only meaningful when compared with a reasonable baseline?
3. If the dataset were 95% one class, would 95% accuracy necessarily be impressive? Explain.

**Your answers:**

# Part III — Visual Data Exploration

We will inspect two features:

- petal length;
- petal width.

The goal is not to prove that the classes are perfectly separable, but to see whether the measurements contain useful information.

In [ ]:
petal_length_idx = feature_names.index("petal length (cm)")
petal_width_idx = feature_names.index("petal width (cm)")

plt.figure(figsize=(8, 5))

for class_id, class_name in enumerate(class_names):
    mask = y == class_id
    plt.scatter(
        X[mask, petal_length_idx],
        X[mask, petal_width_idx],
        label=class_name,
        alpha=0.75
    )

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Iris Classes Using Two Features")
plt.legend()
plt.show()

## Task 3.1 — Interpret the Plot

Answer:

1. Which class appears easiest to separate visually?
2. Which two classes overlap more?
3. Why does visible separation suggest that a classifier may learn useful decision rules?
4. Does a 2D plot show all information in the dataset? Why not?

**Your answers:**

# Part IV — Create Training, Validation, and Test Sets

Lecture 6 distinguishes three roles:

- **training set** — fit model parameters;
- **validation set** — compare model choices and hyperparameters;
- **test set** — estimate final performance after model selection.

For this lab we use:

$$
60\% \text{ training},\qquad
20\% \text{ validation},\qquad
20\% \text{ test}.
$$

We use stratified splitting so that the three flower classes remain approximately balanced.

## Task 4.1 — Predict Split Sizes

The Iris dataset contains 150 examples.

Before running the split, predict approximately:

- **training examples:**  
- **validation examples:**  
- **test examples:**  

Then answer:

1. Which split is used to fit the model?
2. Which split is used to choose `max_depth`?
3. Which split should remain untouched until the final model has been selected?

**Your prediction:**

In [ ]:
# First reserve the final test set: 20% of all data.
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

# Then reserve 25% of the remaining 80% for validation.
# 25% of 80% = 20% of the full dataset.
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval,
)

print("Training examples:", len(X_train))
print("Validation examples:", len(X_val))
print("Test examples:", len(X_test))

## Task 4.2 — Explain `random_state` and `stratify`

Answer:

1. What does `random_state=42` control?
2. Why is reproducibility useful in an educational experiment?
3. What does `stratify` try to preserve?
4. Does the number `42` have any special Machine Learning meaning?

**Your answers:**

# Part V — Train an Initial Decision Tree

A decision tree repeatedly divides the feature space using rules such as:

```text
petal length <= threshold?
```

We start with:

```python
max_depth = 2
```

This is an **initial model**, not yet the final selected model.

## Task 5.1 — Predict Before Training

Before fitting the model:

1. Do you expect training accuracy to be higher, lower, or similar to validation accuracy?
2. Do you expect a depth-2 tree to classify every training example perfectly?
3. Why might limiting tree depth help generalization?
4. Why should we examine **validation** rather than **test** accuracy at this stage?

**Your prediction:**

In [ ]:
model = DecisionTreeClassifier(
    max_depth=2,
    random_state=42
)

model.fit(X_train, y_train)

train_predictions = model.predict(X_train)
val_predictions = model.predict(X_val)

train_accuracy = accuracy_score(y_train, train_predictions)
val_accuracy = accuracy_score(y_val, val_predictions)

print("Training accuracy:", round(train_accuracy, 4))
print("Validation accuracy:", round(val_accuracy, 4))

## Task 5.2 — Training vs. Prediction

Explain what happens in:

```python
model.fit(X_train, y_train)
```

and what happens in:

```python
model.predict(X_val)
```

Be explicit about the difference between:

- **learning rules/parameters** from examples;
- **using the learned model** to predict labels.

**Your answer:**

## Task 5.3 — Interpret Initial Accuracy

Accuracy is

$$
\text{Accuracy}
=
\frac{\text{number of correct predictions}}
{\text{total number of predictions}}.
$$

Answer:

1. What is the training accuracy?
2. What is the validation accuracy?
3. How many validation examples were classified correctly?
4. Is validation accuracy above the majority-class baseline?
5. Why is validation performance more useful than training performance for choosing among model settings?

**Your answers:**

# Part VI — Inspect Validation Predictions

A model evaluation should not stop at one summary number.

At this stage we inspect **validation predictions**, because the validation set is the set used for development and model comparison.

In [ ]:
for i in range(min(10, len(X_val))):
    actual = class_names[y_val[i]]
    predicted = class_names[val_predictions[i]]

    print(
        f"Example {i:>2}: "
        f"actual={actual:<10} "
        f"predicted={predicted:<10} "
        f"{'CORRECT' if actual == predicted else 'WRONG'}"
    )

## Task 6.1 — Explain a Validation Error

If at least one error appears:

1. identify one misclassified example;
2. state its actual class;
3. state its predicted class;
4. explain why a classifier may make errors even when the code is correct.

If no error appears in the first 10 examples, inspect the full validation set.

**Your answer:**

In [ ]:
val_error_indices = np.where(val_predictions != y_val)[0]

print("Number of validation errors:", len(val_error_indices))

for i in val_error_indices:
    print(
        "index =", int(i),
        "| actual =", class_names[y_val[i]],
        "| predicted =", class_names[val_predictions[i]],
        "| features =", X_val[i]
    )

# Part VII — Validation Confusion Matrix

A confusion matrix records how predicted classes correspond to actual classes.

For multiclass classification, entry $(i,j)$ counts examples whose:

- actual class is $i$;
- predicted class is $j$.

At this point we use the **validation set**, not the final test set.

In [ ]:
cm_val = confusion_matrix(y_val, val_predictions)

print(cm_val)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_val,
    display_labels=class_names
)
disp.plot()
plt.title("Confusion Matrix — Validation Set")
plt.show()

## Task 7.1 — Interpret the Validation Confusion Matrix

Answer:

1. Which cells correspond to correct predictions?
2. Which cells correspond to errors?
3. Which pair of classes is most often confused, if any?
4. Why can a confusion matrix reveal information that accuracy alone hides?
5. Why is it acceptable to inspect validation errors during model development?

**Your answers:**

# Part VIII — Model Complexity and Validation-Based Selection

A deeper tree can represent more complicated decision boundaries.

Greater complexity may:

- reduce training error;
- improve validation performance up to a point;
- eventually fit training-specific details rather than general patterns.

This is related to **overfitting**.

We will compare:

```text
max_depth = 1, 2, 3, 5, None
```

using **training and validation data only**.

## Task 8.1 — Predict the Effect of Tree Depth

Before running:

1. Which models do you expect to have the highest training accuracy?
2. Must the model with the highest training accuracy also have the highest validation accuracy?
3. What would a large gap between training and validation accuracy suggest?
4. Why must the test set remain untouched during this comparison?

**Your prediction:**

In [ ]:
depths = [1, 2, 3, 5, None]
depth_results = []

for depth in depths:
    clf = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )
    clf.fit(X_train, y_train)

    train_pred = clf.predict(X_train)
    val_pred = clf.predict(X_val)

    depth_results.append({
        "depth": depth,
        "train_accuracy": accuracy_score(y_train, train_pred),
        "validation_accuracy": accuracy_score(y_val, val_pred),
    })

print("depth | train_acc | val_acc")
for row in depth_results:
    print(
        f"{str(row['depth']):>5} | "
        f"{row['train_accuracy']:.4f} | "
        f"{row['validation_accuracy']:.4f}"
    )

## Task 8.2 — Analyze Complexity

Complete:

| `max_depth` | Training accuracy | Validation accuracy |
|---:|---:|---:|
| 1 |  |  |
| 2 |  |  |
| 3 |  |  |
| 5 |  |  |
| None |  |  |

Then answer:

1. Which setting achieved the highest training accuracy?
2. Which setting achieved the highest validation accuracy?
3. Did increasing complexity always improve validation performance?
4. Which result, if any, provides evidence of underfitting or overfitting?
5. Why should model selection not be based only on training accuracy?

**Your answers:**

## Task 8.3 — Select the Model Using Validation Performance

The selected depth is the one with the highest validation accuracy.

If multiple depths tie, this notebook keeps the first one encountered, which favors the simpler depth earlier in the list.

In [ ]:
best_result = max(
    depth_results,
    key=lambda row: row["validation_accuracy"]
)

selected_depth = best_result["depth"]

print("Selected max_depth:", selected_depth)
print(
    "Best validation accuracy:",
    round(best_result["validation_accuracy"], 4)
)

## Task 8.4 — Training–Validation Generalization Gap

For development purposes, define:

$$
\text{Training–Validation Gap}
=
\text{Training Accuracy}
-
\text{Validation Accuracy}.
$$

A large positive gap can be a warning that the model fits the training data much better than unseen validation examples.

In [ ]:
for row in depth_results:
    gap = (
        row["train_accuracy"]
        - row["validation_accuracy"]
    )
    print(
        f"depth={str(row['depth']):>4} "
        f"| train-val gap={gap:.4f}"
    )

### Interpret the Gap

1. Which model has the largest positive training–validation gap?
2. Is a positive gap automatically proof of severe overfitting?
3. Why can the exact gap change with a different split?
4. Why is validation still only an estimate of future performance?

**Your answers:**

# Part IX — Final Test Evaluation

Only now, after selecting `max_depth` using validation data, do we evaluate on the **test set**.

A common final workflow is:

1. choose the model using training + validation logic;
2. optionally refit the selected model on the combined training and validation data;
3. evaluate once on the untouched test set.

The test set should not influence the choice of `max_depth`.

In [ ]:
X_train_final = np.vstack([X_train, X_val])
y_train_final = np.concatenate([y_train, y_val])

final_model = DecisionTreeClassifier(
    max_depth=selected_depth,
    random_state=42
)

final_model.fit(
    X_train_final,
    y_train_final
)

final_test_predictions = final_model.predict(X_test)
final_test_accuracy = accuracy_score(
    y_test,
    final_test_predictions
)

print("Selected depth:", selected_depth)
print("Final test accuracy:", round(final_test_accuracy, 4))

## Task 9.1 — Interpret the Final Test Result

Answer:

1. What depth was selected using validation data?
2. What is the final test accuracy?
3. Was the test set used to choose the depth?
4. Why is this test result more defensible than testing every depth and choosing the one with the best test score?
5. Why should the final test result still be interpreted as an estimate rather than a guarantee?

**Your answers:**

In [ ]:
cm_test = confusion_matrix(
    y_test,
    final_test_predictions
)

print(cm_test)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_test,
    display_labels=class_names
)
disp.plot()
plt.title("Confusion Matrix — Final Test Set")
plt.show()

test_error_indices = np.where(
    final_test_predictions != y_test
)[0]

print("\nNumber of final test errors:", len(test_error_indices))

for i in test_error_indices:
    print(
        "index =", int(i),
        "| actual =", class_names[y_test[i]],
        "| predicted =", class_names[final_test_predictions[i]],
        "| features =", X_test[i]
    )

## Task 9.2 — Final Error Analysis

1. Which classes, if any, are confused on the final test set?
2. Identify one final test error if one exists.
3. Why is it useful to inspect errors in addition to reporting accuracy?
4. Why should you avoid repeatedly changing the model after seeing final test errors if you still want the test set to remain an unbiased final evaluation?

**Your answers:**

# Part X — Debugging Machine Learning Workflows

## Task 10.1 — Evaluating on Training Data Only

A student reports:

```python
model.fit(X, y)
pred = model.predict(X)
accuracy_score(y, pred)
```

and claims this proves excellent generalization.

1. What is wrong with the evaluation?
2. Which examples were used for both learning and evaluation?
3. Why can this produce an overly optimistic result?
4. What should be done instead?

**Your answer:**

## Task 10.2 — Target Leakage

Suppose a student accidentally adds the true class label as an extra input feature.

1. Why is this called leakage?
2. Why might accuracy become unrealistically high?
3. Would such a model be useful for real prediction when the label is unknown?

**Your answer:**

## Task 10.3 — Fitting on the Test Set

A student repeatedly changes model settings until test accuracy is maximized, then reports that same test accuracy as an unbiased final result.

1. Why has the test set influenced model design?
2. Why is the final test result no longer completely independent?
3. What additional split is often used for model selection?

**Your answer:**

## Task 10.4 — Accuracy Without Context

A classifier achieves 90% accuracy on a dataset where 90% of examples belong to one class.

1. Why may the result be weak?
2. What simple baseline should be checked?
3. What additional evaluation information would help?

**Your answer:**

# Part XI — Personalized Experiment

Use the last digit of your student ID.

- `0–3`: `max_depth = 1`
- `4–6`: `max_depth = 3`
- `7–9`: `max_depth = None`

For this experiment use:

```python
random_state = 7
```

Your assigned depth is fixed **before** seeing the validation or test results. You will still keep the final test set separate.

In [ ]:
LAST_DIGIT = None  # TODO: replace with an integer from 0 to 9

if LAST_DIGIT is not None:
    if 0 <= LAST_DIGIT <= 3:
        PERSONAL_DEPTH = 1
    elif 4 <= LAST_DIGIT <= 6:
        PERSONAL_DEPTH = 3
    elif 7 <= LAST_DIGIT <= 9:
        PERSONAL_DEPTH = None
    else:
        raise ValueError("LAST_DIGIT must be between 0 and 9")

    print("Assigned max_depth:", PERSONAL_DEPTH)

## Task 11.1 — Predict Before Running

Write:

- **My assigned depth:**  
- **Do I expect underfitting, moderate complexity, or high complexity?**  
- **Predicted training accuracy relative to the depth-2 model:**  
- **Predicted validation accuracy relative to the depth-2 model:**  
- **Reason:**  

Only then run the personalized experiment.

In [ ]:
if LAST_DIGIT is not None:
    # 20% final test.
    X_trainval_p, X_test_p, y_trainval_p, y_test_p = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=7,
        stratify=y,
    )

    # 20% validation overall.
    X_train_p, X_val_p, y_train_p, y_val_p = train_test_split(
        X_trainval_p,
        y_trainval_p,
        test_size=0.25,
        random_state=7,
        stratify=y_trainval_p,
    )

    # Development-stage model.
    personal_model = DecisionTreeClassifier(
        max_depth=PERSONAL_DEPTH,
        random_state=7
    )

    personal_model.fit(X_train_p, y_train_p)

    train_pred_p = personal_model.predict(X_train_p)
    val_pred_p = personal_model.predict(X_val_p)

    personal_train_acc = accuracy_score(y_train_p, train_pred_p)
    personal_val_acc = accuracy_score(y_val_p, val_pred_p)

    print("Training accuracy:", round(personal_train_acc, 4))
    print("Validation accuracy:", round(personal_val_acc, 4))

    # Depth was assigned in advance, so now refit on train + validation
    # and evaluate once on the untouched final test set.
    X_fit_p = np.vstack([X_train_p, X_val_p])
    y_fit_p = np.concatenate([y_train_p, y_val_p])

    personal_final_model = DecisionTreeClassifier(
        max_depth=PERSONAL_DEPTH,
        random_state=7
    )
    personal_final_model.fit(X_fit_p, y_fit_p)

    test_pred_p = personal_final_model.predict(X_test_p)
    personal_test_acc = accuracy_score(y_test_p, test_pred_p)

    print("Final test accuracy:", round(personal_test_acc, 4))

## Task 11.2 — Explain the Personalized Result

1. Was your training-accuracy prediction correct?
2. Was your validation-accuracy prediction correct?
3. What was the final test accuracy?
4. Why was the test set not used to choose your assigned depth?
5. What does this experiment teach you about separating **model development** from **final evaluation**?

**Your answers:**

# Part XII — Individual Understanding Check

Your instructor may ask one short question about your notebook.

Possible prompts:

- Show me which array contains the features.
- Show me which array contains the labels.
- What exactly happens during `.fit()`?
- What exactly happens during `.predict()`?
- What is the different role of training, validation, and test data?
- Why must the final test set remain separate during model selection?
- Explain one cell of a confusion matrix.
- What is the majority-class baseline?
- Why can deeper trees overfit?
- What does the training–validation gap mean?
- What changed in your personalized experiment?

> You are expected to explain the **Machine Learning concept represented by the code**, not memorize library syntax.

# Reflection

Answer concisely but precisely.

### R1 — Learning
What does it mean for a model to “learn from data”?

**Answer:**

### R2 — Generalization
Why is performance on unseen data more important than memorizing training examples?

**Answer:**

### R3 — Evaluation
Why should accuracy be interpreted together with class distribution and a confusion matrix?

**Answer:**

### R4 — Complexity
Explain the difference between **underfitting** and **overfitting**.

**Answer:**

### R5 — AI Subdomains
How is this Machine Learning workflow different from the search algorithms studied earlier in the course?

**Answer:**

# Submission Checklist

Before submitting, verify that your notebook contains:

- [ ] feature/target/task-type justification;
- [ ] training-vs-inference explanation;
- [ ] dataset-size and class-balance analysis;
- [ ] visual-data interpretation;
- [ ] train/validation/test split prediction and justification;
- [ ] trained initial decision-tree model;
- [ ] training and validation accuracy interpretation;
- [ ] validation prediction-error analysis;
- [ ] validation confusion-matrix interpretation;
- [ ] model-depth experiment using validation data;
- [ ] selected depth justified from validation performance;
- [ ] training–validation gap analysis;
- [ ] final test evaluation performed only after model selection;
- [ ] final test confusion-matrix/error interpretation;
- [ ] debugging/workflow answers;
- [ ] personalized model experiment;
- [ ] prediction before personalized execution;
- [ ] reflection answers;
- [ ] visible outputs from important code cells.

Suggested filename:

```text
Lab11_StudentID.ipynb
```

# Assessment Guide — 10 Marks

| Component | Marks | Evidence expected |
|---|---:|---|
| **Correct implementation** | **2.0** | Valid train/validation/test pipeline, training, prediction, and final evaluation |
| **Conceptual / algorithmic justification** | **3.0** | Explains features/labels, training/inference, split roles, model selection, generalization, and complexity |
| **Experimental analysis** | **2.0** | Interprets validation results, confusion matrices, depth experiment, and final test performance |
| **Prediction / debugging / reasoning** | **1.0** | Predictions and diagnosis of leakage/test-tuning/evaluation errors |
| **Individual understanding check** | **1.0** | Short explanation of selected part of the student's own work |
| **Code quality & completeness** | **1.0** | Readable code, complete responses, required outputs |
| **Total** | **10.0** |  |

> **Key rule:** A high accuracy without a correct explanation of the evaluation methodology earns only limited credit.

## Key Takeaways

- Supervised learning uses examples $(x_i,y_i)$ to learn a predictive model.
- Features describe the input; targets are the desired outputs.
- Training and inference are different phases.
- **Training data** fits model parameters.
- **Validation data** supports model/hyperparameter selection.
- **Test data** is reserved for final evaluation after model selection.
- Accuracy should be interpreted relative to a baseline and alongside a confusion matrix.
- A model can fit training data very well but generalize poorly.
- Model complexity affects the balance between underfitting and overfitting.
- A valid Machine Learning result depends on both the model and the **evaluation methodology**.

The next lab will introduce compact practical examples from **Natural Language Processing and Computer Vision**.